# C3 Mirror Cluster Inspector

**Cluster C3**: 14 tasks from the scene-first LLM clustering.
- 12 core: standard **mirror-reflection / append** tasks (input → 2× or 4× larger output)
- 2 outliers (marked): **symmetry restoration** — fill a masked hole in an already-symmetric pattern (same-size input/output)

Grids load from local `data/training/` if accessible, else `/tmp/arc-data/data/training` (ARC-AGI clone).

In [1]:
%matplotlib inline
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

def _find_data_dir():
    '''Walk up from cwd; test that a probe file is actually readable.'''
    probe = '4c4377d9.json'
    candidates = []
    p = Path.cwd()
    for _ in range(8):
        td = p / 'data' / 'training'
        if td.exists():
            candidates.append(td)
        p = p.parent
    candidates.append(Path('/tmp/arc-data/data/training'))  # fallback
    for td in candidates:
        try:
            (td / probe).read_text()  # will raise if inaccessible
            print(f'Using data dir: {td}')
            return td
        except Exception:
            continue
    raise FileNotFoundError(
        'Could not find readable ARC training data. '
        'Run: git clone --depth=1 https://github.com/fchollet/ARC-AGI.git /tmp/arc-data'
    )

DATA_DIR = _find_data_dir()

def show_grid(ax, grid, title, highlight=False):
    ax.imshow(np.array(grid), cmap=CMAP, norm=NORM, interpolation='nearest')
    ax.set_title(title, fontsize=8)
    ax.axis('off')
    if highlight:
        for spine in ax.spines.values():
            spine.set_edgecolor('#8B0000')
            spine.set_linewidth(2)

Using data dir: /Users/rodhyde/Documents/GitHub/arc-agi-solver/data/training


In [ ]:
# C3 task list (scene-first cluster, 14 tasks)
OUTLIER_IDS = {'3631a71a', 'b8825c91'}

C3_TASKS = {
    "3af2c5a8": "TYPE: mirror-append (H+V)\nMECHANISM: Output = 2x2 tiling by reflecting input across both axes.\nGrid: 3x4 \u2192 6x8",
    "49d1d64f": "TYPE: mirror-append (H+V)\nMECHANISM: Output = 2x2 tiling by reflecting across both axes.\nGrid: 2x2 \u2192 4x4",
    "4c4377d9": "TYPE: vertical mirror append\nMECHANISM: Output = input stacked above its vertical flip.\nGrid: 3x4 \u2192 6x4",
    "62c24649": "TYPE: mirror-append (H+V)\nMECHANISM: Reflection across both axes to make 2x2 symmetric tile.\nGrid: 3x3 \u2192 6x6",
    "67e8384a": "TYPE: mirror-append (H+V)\nMECHANISM: 2x2 symmetric tiling via double reflection.\nGrid: 3x3 \u2192 6x6",
    "6d0aefbc": "TYPE: horizontal mirror append\nMECHANISM: Input placed left; left-right flip placed right.\nGrid: 3x3 \u2192 3x6",
    "6fa7a44f": "TYPE: vertical mirror append\nMECHANISM: Input on top; top-bottom flip below.\nGrid: 3x3 \u2192 6x3",
    "7fe24cdd": "TYPE: mirror-append (H+V)\nMECHANISM: 2x2 symmetric tiling.\nGrid: 3x3 \u2192 6x6",
    "8be77c9e": "TYPE: vertical mirror append\nMECHANISM: Stack input above its vertical flip.\nGrid: 3x3 \u2192 6x3",
    "8d5021e8": "TYPE: mirror-append (H+V)\nMECHANISM: 3x2 tiling by reflection? Input 3x2 \u2192 9x4 (unusual ratio \u2014 worth checking).\nGrid: 3x2 \u2192 9x4",
    "a416b8f3": "TYPE: horizontal mirror append\nMECHANISM: Append left-right flip of input to the right.\nGrid: 3x3 \u2192 3x6",
    "c9e6f938": "TYPE: horizontal mirror append\nMECHANISM: Append left-right flip of input to the right.\nGrid: 3x3 \u2192 3x6",
    "3631a71a": "*** OUTLIER \u2014 SYMMETRY RESTORATION ***\nTYPE: symmetry restoration via mask replacement\nSCENE: Large 30x30 grid. The pattern is bilaterally symmetric, but a rectangular hole is filled with 9s (the mask colour). The non-9 cells clearly reveal the symmetry axis.\nMECHANISM: Identify the symmetry axis of the non-9 region. For each masked 9 cell, find its symmetric counterpart and copy its colour.\nKEY DIFFERENCE: This is the INVERSE of mirror-append \u2014 instead of creating a reflection, it FILLS IN a missing part of an already-symmetric pattern.",
    "b8825c91": "*** OUTLIER \u2014 SYMMETRY RESTORATION ***\nTYPE: symmetric pattern restoration\nSCENE: 16x16 grid with a symmetric coloured pattern and a hole (9s).\nMECHANISM: Detect symmetry axis, fill masked region by reading the symmetric counterpart cells.\nKEY DIFFERENCE: Same inverse-mirror operation as 3631a71a \u2014 restores a hole in an existing symmetric pattern.",
}

# Show core tasks first, outliers last
task_ids = sorted(C3_TASKS.keys(), key=lambda t: (t in OUTLIER_IDS, t))
print(f'C3 cluster: {len(task_ids)} tasks')
print(f'  Core mirror tasks  : {sum(1 for t in task_ids if t not in OUTLIER_IDS)}')
print(f'  Outliers (restore) : {sum(1 for t in task_ids if t in OUTLIER_IDS)}')

In [ ]:
for tid in task_ids:
    desc = C3_TASKS[tid]
    is_outlier = tid in OUTLIER_IDS

    task_path = DATA_DIR / f'{tid}.json'
    if not task_path.exists():
        print(f'!! {tid}: file not found at {task_path}')
        continue
    task  = json.loads(task_path.read_text())
    pairs = task['train']
    n_rows = len(pairs) + 1

    marker = '  *** OUTLIER ***' if is_outlier else ''
    print(f"\n{'='*60}")
    print(f'  {tid}  ({len(pairs)} train pairs){marker}')
    print(f"{'='*60}")

    fig, axes = plt.subplots(n_rows, 2, figsize=(6, 2.5 * n_rows), squeeze=False)
    if is_outlier:
        fig.patch.set_facecolor('#fff0f0')
    title_color = '#8B0000' if is_outlier else '#003366'
    fig.suptitle(f'{tid}{marker}', fontsize=9, color=title_color, y=1.01)

    for i, pair in enumerate(pairs):
        show_grid(axes[i, 0], pair['input'],  f'Train {i+1} in',  highlight=is_outlier)
        show_grid(axes[i, 1], pair['output'], f'Train {i+1} out', highlight=is_outlier)

    test = task['test'][0]
    show_grid(axes[-1, 0], test['input'], 'Test in', highlight=is_outlier)
    if 'output' in test:
        show_grid(axes[-1, 1], test['output'], 'Test out', highlight=is_outlier)
    else:
        axes[-1, 1].axis('off')
        axes[-1, 1].set_title('Test out (hidden)', fontsize=8)

    plt.tight_layout()
    plt.show()

    for line in desc.split('\n'):
        print(f'  {line}')
    print()